# Fake News Detection Using Metadata and NLP Features

This project builds an SVM-based fake news classifier using a combination
of article metadata and high-dimensional NLP features.

The workflow includes:
- Data cleaning
- Feature engineering
- Categorical encoding
- PCA dimensionality reduction
- SVM classification
- Target-proxy analysis

In [1]:
import numpy as np
import pandas as pd 
import scipy.sparse as ss

from sklearn.model_selection import train_test_split

from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, f1_score

In [2]:
train = pd.read_csv('news_train.csv')
train_vectors = ss.load_npz('news_train_text_vectors.npz')

test = pd.read_csv('news_test.csv')
test_vectors = ss.load_npz('news_test_text_vectors.npz')

In [3]:
print("train metadata:", train.shape)
print("train text vectors:", train_vectors.shape)

print("test metadata:", test.shape)
print("test text vectors:", test_vectors.shape)

train metadata: (1500, 5)
train text vectors: (1500, 42141)
test metadata: (346, 4)
test text vectors: (346, 42141)


In [4]:
train_text = pd.DataFrame.sparse.from_spmatrix(train_vectors)
test_text = pd.DataFrame.sparse.from_spmatrix(test_vectors)

In [5]:
print(train_text.shape)
print(test_text.shape)

(1500, 42141)
(346, 42141)


## dropping duplicates

In [6]:
# duplicate rows in train
dup_idx = train[train.duplicated()].index

# remove same rows from both
train = train.drop(index=dup_idx).reset_index(drop=True)
train_text = train_text.drop(index=dup_idx).reset_index(drop=True)

In [7]:
print(train.duplicated().sum(), train_text.duplicated().sum())

0 18


In [8]:
print(train.shape[0] == train_text.shape[0])
print(test.shape[0] == test_text.shape[0])

True
True


## handling missing values

In [9]:
print(train.isna().sum())
print(test.isna().sum())

author       0
published    0
site_url     0
type         0
label        0
dtype: int64
author       0
published    0
site_url     0
type         0
dtype: int64


In [10]:
print(train_text.isna().sum().sum())
print(test_text.isna().sum().sum())

42141
42141


In [11]:
train_text = train_text.fillna(0)
test_text = test_text.fillna(0)

In [12]:
print(train_text.isna().sum().sum())
print(test_text.isna().sum().sum())

0
0


# splitting

In [13]:
y = train["label"]
y = y.map({
    "Real": 0,
    "Fake": 1})

train_meta = train.drop(columns=["label"])

In [14]:
print(y.value_counts())

label
1    955
0    512
Name: count, dtype: int64


In [15]:
from sklearn.model_selection import train_test_split

x_train, x_val, text_train, text_val, y_train, y_val = train_test_split(train_meta, train_text, y, test_size=0.2, stratify=y, random_state=42)

# feature engineering

## date time

In [16]:
for df in [x_train, x_val, test]:
    df["published"] = pd.to_datetime(
        df["published"],
        errors="coerce",
        utc=True
    )

    df["year"] = df["published"].dt.year
    df["month"] = df["published"].dt.month
    df["day"] = df["published"].dt.day
    df["dayofweek"] = df["published"].dt.dayofweek
    df["hour"] = df["published"].dt.hour

    df.drop(columns=["published"], inplace=True)

## handling categorical features

In [17]:
x_train.head()

,author,site_url,type,year,month,day,dayofweek,hour
7,Fed Up,100percentfedup.com,bias,2016.0,11.0,22.0,1.0,19.0
1116,-NO AUTHOR-,wnd.com,bias,2016.0,10.0,26.0,2.0,21.0
439,No Author,awdnews.com,conspiracy,2016.0,10.0,30.0,6.0,11.0
974,Rhonda Gilmore,westernjournalism.com,bias,2016.0,10.0,26.0,2.0,19.0
33,Alex Jones,infowars.com,conspiracy,2016.0,10.0,27.0,3.0,15.0


In [18]:
x_train["author"].value_counts().head(10)

author
No Author               257
Activist Post            55
EdJenner                 45
Daniel Greenfield        32
Jason Easley             31
admin                    31
Alex Ansary              26
Dr. Patrick Slattery     23
Anonymous                21
Henry Wolff              17
Name: count, dtype: int64

In [19]:
x_train["site_url"].value_counts().head(10)

site_url
westernjournalism.com    71
prisonplanet.com         70
infowars.com             68
activistpost.com         68
returnofkings.com        67
awdnews.com              65
frontpagemag.com         62
politicususa.com         61
clickhole.com            60
naturalnews.com          59
Name: count, dtype: int64

In [20]:
x_train["type"].value_counts()

type
bs            311
conspiracy    292
bias          254
hate          155
satire         90
junksci        60
fake           11
Name: count, dtype: int64

In [21]:
# Frequency encoding - learn only from x_train
for col in ["author", "site_url"]:
    freq = x_train[col].value_counts()

    x_train[col + "_freq"] = x_train[col].map(freq)
    x_val[col + "_freq"] = x_val[col].map(freq).fillna(0)
    test[col + "_freq"] = test[col].map(freq).fillna(0)

# remove original high-cardinality columns
x_train.drop(columns=["author", "site_url"], inplace=True)
x_val.drop(columns=["author", "site_url"], inplace=True)
test.drop(columns=["author", "site_url"], inplace=True)

In [22]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

type_train = encoder.fit_transform(x_train[["type"]])
type_val = encoder.transform(x_val[["type"]])
type_test = encoder.transform(test[["type"]])

x_train = x_train.drop(columns=["type"])
x_val = x_val.drop(columns=["type"])
test = test.drop(columns=["type"])

In [23]:
type_cols = encoder.get_feature_names_out(["type"])

type_train = pd.DataFrame(type_train, columns=type_cols, index=x_train.index)
type_val = pd.DataFrame(type_val, columns=type_cols, index=x_val.index)
type_test = pd.DataFrame(type_test, columns=type_cols, index=test.index)

x_train = pd.concat([x_train, type_train], axis=1)
x_val = pd.concat([x_val, type_val], axis=1)
test = pd.concat([test, type_test], axis=1)

In [24]:
x_train.head()

,year,month,day,dayofweek,hour,author_freq,site_url_freq,type_bias,type_bs,type_conspiracy,type_fake,type_hate,type_junksci,type_satire
7,2016.0,11.0,22.0,1.0,19.0,8,23,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1116,2016.0,10.0,26.0,2.0,21.0,11,31,1.0,0.0,0.0,0.0,0.0,0.0,0.0
439,2016.0,10.0,30.0,6.0,11.0,257,65,0.0,0.0,1.0,0.0,0.0,0.0,0.0
974,2016.0,10.0,26.0,2.0,19.0,2,71,1.0,0.0,0.0,0.0,0.0,0.0,0.0
33,2016.0,10.0,27.0,3.0,15.0,2,68,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [25]:
x_train.isna().sum()

year               1
month              1
day                1
dayofweek          1
hour               1
author_freq        0
site_url_freq      0
type_bias          0
type_bs            0
type_conspiracy    0
type_fake          0
type_hate          0
type_junksci       0
type_satire        0
dtype: int64

# PCA

In [26]:
from sklearn.decomposition import PCA

pca = PCA(n_components=200, random_state=42)

text_train_pca = pca.fit_transform(text_train)
text_val_pca = pca.transform(text_val)
test_text_pca = pca.transform(test_text)

In [27]:
print(text_train_pca.shape)
print(text_val_pca.shape)
print(test_text_pca.shape)

(1173, 200)
(294, 200)
(346, 200)


In [28]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

x_train_clean = imputer.fit_transform(x_train)
x_val_clean = imputer.transform(x_val)
test_clean = imputer.transform(test)

In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_final = scaler.fit_transform(
    np.hstack([x_train_clean, text_train_pca])
)
X_val_final = scaler.transform(
    np.hstack([x_val_clean, text_val_pca])
)
X_test_final = scaler.transform(
    np.hstack([test_clean, test_text_pca])
)

In [30]:
print(
    "Explained variance:",
    pca.explained_variance_ratio_.sum()
)

Explained variance: 0.37422159901388513


# training the model

In [31]:
model = SVC(
    kernel="linear",
    C=1
)

model.fit(X_train_final, y_train)

y_pred = model.predict(X_val_final)

print("F1:", f1_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

F1: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       103
           1       1.00      1.00      1.00       191

    accuracy                           1.00       294
   macro avg       1.00      1.00      1.00       294
weighted avg       1.00      1.00      1.00       294



# Target Proxy / Leakage Analysis

The `type` feature perfectly separates the target classes in the training data. Because it acts as a target proxy and produces an unrealistically perfect validation score, the final portfolio model excludes `type`.


In [32]:
pd.crosstab(train["type"], train["label"])

label,Fake,Real
type,,
bias,0,315
bs,396,0
conspiracy,358,0
fake,13,0
hate,0,197
junksci,76,0
satire,112,0


In [33]:
pd.crosstab(
    train["type"],
    train["label"],
    normalize="index"
)

label,Fake,Real
type,,
bias,0.0,1.0
bs,1.0,0.0
conspiracy,1.0,0.0
fake,1.0,0.0
hate,0.0,1.0
junksci,1.0,0.0
satire,1.0,0.0


## Final Model Without the `type` Feature

In [34]:
type_cols = [col for col in x_train.columns if col.startswith("type_")]

x_train_no_type = x_train.drop(columns=type_cols)
x_val_no_type = x_val.drop(columns=type_cols)
test_no_type = test.drop(columns=type_cols)

In [35]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

x_train_clean = imputer.fit_transform(x_train_no_type)
x_val_clean = imputer.transform(x_val_no_type)
test_clean = imputer.transform(test_no_type)

In [36]:
X_train_final = np.hstack([
    x_train_clean,
    text_train_pca
])

X_val_final = np.hstack([
    x_val_clean,
    text_val_pca
])

X_test_final = np.hstack([
    test_clean,
    test_text_pca
])

In [37]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_final = scaler.fit_transform(X_train_final)
X_val_final = scaler.transform(X_val_final)
X_test_final = scaler.transform(X_test_final)

In [38]:
validation_model = SVC(
    kernel="linear",
    C=1,
    class_weight="balanced"
)

validation_model.fit(X_train_final, y_train)

y_pred = validation_model.predict(X_val_final)

print("F1:", f1_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

F1: 0.8498727735368957
              precision    recall  f1-score   support

           0       0.74      0.66      0.70       103
           1       0.83      0.87      0.85       191

    accuracy                           0.80       294
   macro avg       0.78      0.77      0.77       294
weighted avg       0.80      0.80      0.80       294



## Final Training on the Full Training Set

In [39]:
X_full_final = np.vstack([
    X_train_final,
    X_val_final
])

y_full = pd.concat([
    y_train,
    y_val
]).to_numpy()

In [40]:
print("Full training features:", X_full_final.shape)
print("Full training labels:", y_full.shape)

Full training features: (1467, 207)
Full training labels: (1467,)


In [41]:
model = SVC(
    kernel="linear",
    C=1,
    class_weight="balanced"
)

model.fit(X_full_final, y_full)

,C,1
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [42]:
test_pred = model.predict(X_test_final)

submission = pd.DataFrame({
    "label": np.where(
        test_pred == 1,
        "Fake",
        "Real"
    )
})

print(submission.shape)
submission.head()

(346, 1)


,label
0,Real
1,Fake
2,Real
3,Real
4,Fake


## Submission Artifacts

This cell exports the trained SVC, prediction file, and notebook in the format required by the course submission workflow.


In [44]:
import zipfile
import joblib

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

joblib.dump(model, 'model')
submission.to_csv('submission.csv', index=False)
file_names = ['fake_news_detection.ipynb', 'submission.csv', 'model']
compress(file_names)

File Paths:
['fake_news_detection.ipynb', 'submission.csv', 'model']


## Results

| Experiment | F1 |
| --- | ---: |
| SVC with `type` | 1.000 |
| SVC without `type` + balanced weights | 0.850 |

The final model is the balanced linear SVC without the `type` feature.
